## Importation des données

In [1]:
import requests
import pandas as pd

# Fonction permettant de récupérer les données sur l'API de l'ADEME
def get_dpe(type):
    if type == "existant":
        url = "https://data.ademe.fr/data-fair/api/v1/datasets/dpe03existant/lines"
    elif type == "neuf":
        url = "https://data.ademe.fr/data-fair/api/v1/datasets/dpe02neuf/lines"
    else:
        print("Type de logement introuvable")
        return None

    params = {
        "qs": "code_region_ban:94", # On se base dans la région CORSE
        "size": "500"
    }

    toutes_les_lignes = []

    while url:       
        response = requests.get(url, params=params)
        data = response.json()

        # Pour afficher le nombre total de ligne à scrapper 
        if params != None :
            print(f"Nombre total de ligne : {data.get('total')}")

        toutes_les_lignes.extend(data["results"])

        print(f"Nombre de lignes faites : {len(toutes_les_lignes)}")

        url = data.get("next")

        params = None

    df = pd.DataFrame(toutes_les_lignes)
    df.to_csv(f"Data/dpe_{type}.csv", index=False, encoding="utf-8")
    return df

In [2]:
import os
os.chdir(r"../")
os.listdir()

#Importation des nouveaux logements
if not os.path.exists("Data/dpe_neuf.csv") :
    dpe_neuf = get_dpe("neuf")
else :
    dpe_neuf = pd.read_csv("Data/dpe_neuf.csv", header=0, encoding="utf-8", low_memory=False)

#Importation des logements existant
if not os.path.exists("Data/dpe_existant.csv") :
    dpe_existant = get_dpe("existant")
else :
    dpe_existant = pd.read_csv("Data/dpe_existant.csv", header=0, encoding="utf-8", low_memory=False)

### Fusion des deux dataframe pour en avoir qu'un seul

In [3]:
dpe_neuf = dpe_neuf.assign(logement="neuf", annee_construction=2026)
dpe_existant = dpe_existant.assign(logement="ancien")

column_commune = dpe_neuf.columns.intersection(dpe_existant.columns)

data = pd.concat([dpe_neuf.loc[:,column_commune],dpe_existant.loc[:,column_commune]], ignore_index=True)
data.shape

C:\Users\nijim\AppData\Local\Temp\ipykernel_16488\3889685206.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dpe_existant = dpe_existant.assign(logement="ancien")


(59570, 175)

### Passage du numero_dpe en index de dataframe

In [4]:
data.index = data["numero_dpe"]
data = data.drop(columns="numero_dpe")
data.shape

(59570, 174)

### Enregistrement des différentes variables (cible et illustrative)

In [5]:
Variables_cible = data.filter(regex="^conso_").columns.to_list() # Toute les colone lié à la conssomation
Variables_cible.extend(data.filter(regex="^cout_").columns.to_list())
Variables_cible.extend("etiquette_dpe")
print(Variables_cible)

['conso_auxiliaires_ep', 'conso_auxiliaires_ef', 'conso_5_usages_par_m2_ep', 'conso_ecs_ef_energie_n1', 'conso_5_usages_par_m2_ef', 'conso_chauffage_ef_energie_n1', 'conso_5_usages_ef', 'conso_5_usages_ef_energie_n1', 'conso_5_usages_ep', 'conso_chauffage_ef', 'conso_ecs_ep', 'conso_ecs_ef', 'conso_ef_installation_ecs_n1', 'conso_refroidissement_ef', 'conso_eclairage_ef', 'conso_chauffage_ep', 'conso_eclairage_ep', 'conso_refroidissement_ep', 'conso_ecs_ef_energie_n2', 'conso_chauffage_ef_energie_n2', 'conso_5_usages_ef_energie_n2', 'conso_chauffage_installation_chauffage_n1', 'conso_chauffage_installation_chauffage_n2', 'cout_eclairage', 'cout_chauffage', 'cout_total_5_usages_energie_n1', 'cout_ecs_energie_n1', 'cout_ecs', 'cout_auxiliaires', 'cout_total_5_usages', 'cout_refroidissement', 'cout_chauffage_energie_n1', 'cout_ecs_energie_n2', 'cout_total_5_usages_energie_n2', 'cout_chauffage_energie_n2', 'e', 't', 'i', 'q', 'u', 'e', 't', 't', 'e', '_', 'd', 'p', 'e']


In [6]:
Variables_illustrative = ["coordonnee_cartographique_x_ban","coordonnee_cartographique_y_ban","_geopoint","etiquette_ges"]

## Vérification des types des variables

In [7]:
#Variable booléenne
#Si min = 0 et max = 1 -> c'est du booléen
min0 = data.min()==0
max1 = data.max()==1

bool_cols = data.columns[min0 & max1]

data[bool_cols] = data[bool_cols].astype(bool)

data.select_dtypes(include="bool").info()

<class 'pandas.DataFrame'>
Index: 59570 entries, 2620N0025074S to 2620E2590403W
Data columns (total 7 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   ventilation_posterieure_2012   59570 non-null  bool 
 1   appartement_non_visite         59570 non-null  bool 
 2   presence_brasseur_air          59570 non-null  bool 
 3   isolation_toiture              59570 non-null  bool 
 4   protection_solaire_exterieure  59570 non-null  bool 
 5   logement_traversant            59570 non-null  bool 
 6   inertie_lourde                 59570 non-null  bool 
dtypes: bool(7)
memory usage: 1.6 MB


In [8]:
#Variable quali avec un codage numérique -> obligé de codé en dur
data["version_dpe"] = data["version_dpe"].astype(str)
data["code_postal_ban"] = data["code_postal_ban"].astype(str)
data["code_postal_brut"] = data["code_postal_brut"].astype(str)

In [9]:
#Variable date
cols_date = data.columns[data.columns.str.contains("date")]
data[cols_date] = data[cols_date].apply(pd.to_datetime, errors="coerce")

## Sélection de variables

In [10]:
col_a_sup = [] # Pour garder toutes les variables à supprimé

### Variable avec trop de valeurs manquantes

In [11]:
# Si plus de 50% de valeurs manquantes -> On supprime
na = data.columns[(data.isna().sum()*100/data.shape[0])>=50]

col_a_sup.extend(na)
print(len(na))

41


### Suppréssion des variables unique/dpe ET identique pour tout les dpe

In [12]:
#Si la variable est identique partout
identique = data.nunique()==1
not_this_col = data.columns[identique] #On récupère son nom
col_a_sup.extend(not_this_col)
print("Identiques : ",not_this_col)

#Si la variable est différente partout
unique = data.nunique()==data.shape[0]
not_this_col = data.columns[unique] #On récupère son nom
col_a_sup.extend(not_this_col)
print("Unique : ", not_this_col)

Identiques :  Index(['code_region_ban'], dtype='str')
Unique :  Index(['_i', '_id'], dtype='str')


### Suppréssion des variables qualitatives avec trop de modalité

In [13]:
import numpy as np
print(np.sort(data.select_dtypes(include="str").nunique())) 

[    2     2     2     2     2     2     2     3     3     3     3     3
     3     3     3     3     3     4     4     4     4     4     4     4
     4     4     5     6     7     7     7     8     8     9     9    10
    10    10    10    10    10    11    12    13    17    31    38    44
    57    71    79    85    85   106   110   112   117   342   345   358
   486   508   927   956  1915  1980  2107  3562  4566  6302 13446 13593
 14705 18258 28056 30529 33025 59570]


On remqarque qu'après 117 on passe comme un cap, on va se fixé 150 comme nombre de modalité maximal

In [14]:
trop_quali = data.select_dtypes(include="str").nunique()>=150

col_a_sup.extend(data.select_dtypes(include="str").columns[trop_quali])
print(len(trop_quali))

78


### Supprésion des variables avec une trop faible variance 

In [15]:
from sklearn.feature_selection import VarianceThreshold
transformer = VarianceThreshold(threshold=0)
transformer.fit_transform(data.select_dtypes(include="float"))

col_a_sup.extend(data.select_dtypes(include="float").columns[~transformer.get_support()]) 

c:\Users\nijim\Documents\GitHub\Machine-Learning-sous-Python\.venv\Lib\site-packages\sklearn\feature_selection\_variance_threshold.py:114: RuntimeWarning: Degrees of freedom <= 0 for slice.
  self.variances_ = np.nanvar(X, axis=0)
c:\Users\nijim\Documents\GitHub\Machine-Learning-sous-Python\.venv\Lib\site-packages\sklearn\feature_selection\_variance_threshold.py:122: RuntimeWarning: All-NaN slice encountered
  self.variances_ = np.nanmin(compare_arr, axis=0)


### Netoyage des colones à supprimer

In [16]:
col_a_sup = [x for x in set(col_a_sup) if (x not in Variables_cible) and (x not in Variables_illustrative)]

print(len(col_a_sup))
print(col_a_sup)

50
['code_region_ban', 'description_installation_chauffage_n2', 'nom_commune_ban', 'emission_ges_ecs_energie_n2', 'identifiant_ban', 'surface_tertiaire_immeuble', 'id_rnb', 'type_emetteur_installation_chauffage_n2', '_id', 'type_generateur_chauffage_principal', 'qualite_isolation_plancher_haut_comble_amenage', 'type_generateur_n1_installation_n2', 'emission_ges_chauffage_energie_n2', 'complement_adresse_batiment', 'categorie_enr', 'usage_generateur_n2_installation_n1', '_score', 'code_insee_ban', 'nom_commune_brut', 'numero_dpe_remplace', 'nombre_niveau_immeuble', 'position_logement_dans_immeuble', 'type_generateur_froid', 'description_generateur_froid', 'code_postal_brut', 'surface_chauffee_installation_chauffage_n2', 'typologie_logement', 'usage_generateur_n1_installation_n2', 'description_installation_ecs_n1', 'nom_rue_ban', 'surface_habitable_immeuble', 'adresse_complete_brut', 'numero_dpe_immeuble_associe', 'adresse_ban', 'provenance_id_rnb', 'emission_ges_5_usages_energie_n2', 'n

In [17]:
data = data.drop(columns=col_a_sup)
print(data.shape)

(59570, 124)


In [18]:
colone_explicative = [c for c in data.columns if c not in Variables_cible and c not in Variables_illustrative]
print(len(colone_explicative))
print(colone_explicative)

85
['configuration_installation_chauffage_n1', 'type_generateur_n1_ecs_n1', 'score_ban', 'deperditions_murs', 'statut_geocodage', 'ventilation_posterieure_2012', 'date_etablissement_dpe', 'emission_ges_chauffage', 'qualite_isolation_menuiseries', 'date_reception_dpe', 'qualite_isolation_plancher_bas', 'modele_dpe', 'qualite_isolation_enveloppe', 'type_energie_n1', 'emission_ges_eclairage', 'code_postal_ban', 'emission_ges_ecs', 'deperditions_planchers_bas', 'date_fin_validite_dpe', 'deperditions_enveloppe', 'volume_stockage_generateur_n1_ecs_n1', 'version_dpe', 'besoin_ecs', 'type_energie_principale_ecs', 'deperditions_renouvellement_air', '_rand', 'surface_habitable_desservie_par_installation_ecs_n1', 'production_electricite_pv_kwhep_par_an', 'nombre_niveau_logement', 'surface_habitable_logement', 'type_installation_ecs_n1', 'emission_ges_5_usages_energie_n1', 'emission_ges_5_usages_par_m2', 'type_generateur_n1_installation_n1', 'nombre_logements_desservis_par_installation_ecs_n1', 't

In [19]:
#A servi à analysé les colones restantes
for col in colone_explicative:
    if col not in Variables_cible and col not in Variables_illustrative:
        print(f"----{col}----")
        print(data[col])
        print(len(data[col].unique()))
        print("--------------")

----configuration_installation_chauffage_n1----
numero_dpe
2620N0025074S    Installation de chauffage simple
2620N0011394O                                 NaN
2420N3326336G                                 NaN
2420N0067785J                                 NaN
2120N0386025G                                 NaN
                               ...               
2620E2293727G    Installation de chauffage simple
2620E2378054P    Installation de chauffage simple
2620E2410950V    Installation de chauffage simple
2620E2547824F    Installation de chauffage simple
2620E2590403W    Installation de chauffage simple
Name: configuration_installation_chauffage_n1, Length: 59570, dtype: str
11
--------------
----type_generateur_n1_ecs_n1----
numero_dpe
2620N0025074S                     CET sur air extérieur après 2014
2620N0011394O                                                  NaN
2420N3326336G                                                  NaN
2420N0067785J                                         

## Suppression des outliers

## Enregistrement du dataframe final

In [20]:
data.to_csv("Data/dpe.csv", index=True, encoding="utf-8")